In [1]:
## Initialize Hyperparameters and import libraries

import numpy as np
import torch

from model import *
from utilities import *
from loss_ftn import *

In [ ]:
PATH = "D:\\01_Datasets\\Curriculum Learning\\"#true_results\\result_Hy_"+f'{n:08d}'+'gamma'+str(0.0).zfill(2)+".npy"

data_dims = np.shape(np.load(PATH + "true_results_multiwl\\result_Hy_"+f'{0:08d}'+'k'+str(0.01).zfill(4)+".npy")[0])

In [3]:
n_train = 4500
n_test = 1000

batch_size = 250
epochs = 100

name = (n_train/1000)

layer_finetune = 10

In [4]:
train_loader ,test_loader = data_loader(n_train, n_test, batch_size, wl_list, data_dims, 0, dz)

Loading Test Data: 100%|██████████| 1000/1000 [01:01<00:00, 16.30it/s]


In [5]:
model = FNOModel2d(modes=16, width=32, blocks=layer_finetune).cuda()

In [6]:
lr_top = 5e-5
step_size = 30
gamma = 0.5

In [7]:
name = (n_train/1000)
SAVE_pretrain = f"DIRTL_{name:.2f}k_pretrain_{layer_finetune}FL.pth"
SAVE_model = f"DIRTL_{name * 2 :.2f}k_loose_{layer_finetune}FL.pth"
SAVE_lc =  f"DIRTL_{name* 2 :.2f}k_loose_{layer_finetune}FL_LC.npz"

In [8]:
model.load_state_dict(torch.load(SAVE_pretrain))

<All keys matched successfully>

In [9]:
optimizer = torch.optim.Adam(model.parameters(), lr=lr_top)

Train_rel_L1_arr = []
Train_rel_L2_arr = []
Test_rel_L1_arr = []
Test_rel_L2_arr = []

# Define StepLR scheduler
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=step_size,    # number of epochs before decay
    gamma=gamma             # decay factor, e.g., 0.1 reduces LR by 10x
)

loss = nn.MSELoss()

# gc.collect(k)
torch.cuda.empty_cache()

total_time = 0

for ep in range(epochs):
    t1 = default_timer()
    model.train()
    Train_mse = 0

    for input_shape, result in train_loader:
        input_shape, result = input_shape.cuda(), result.cuda()
        optimizer.zero_grad()
        
        out = model((input_shape))

        Train_mse_temp = loss(out.reshape(batch_size, -1), result.reshape(batch_size, -1))

        Train_mse_temp.backward()
        
        optimizer.step()
        
        Train_mse += Train_mse_temp.detach() * batch_size

    scheduler.step()

    model.eval()
    Test_mse = 0.0
    with torch.no_grad():
        for input_shape, result in test_loader:
            input_shape, result = input_shape.cuda(), result.cuda()

            out = model((input_shape))
            Test_mse_temp = loss(out.reshape(batch_size, -1), result.reshape(batch_size, -1))
            Test_mse += Test_mse_temp.detach() * batch_size

    Train_mse /= len(train_loader.dataset)
    Test_mse /= len(test_loader.dataset)

    Train_rmse = np.sqrt(Train_mse.item())
    Test_rmse = np.sqrt(Test_mse.item())

    if SAVE_lc:
        with torch.no_grad():
            model.eval()
            _, _, rel_L1, rel_L2, _ = rel_err(model, train_loader)
            Train_rel_L1_arr.append(np.mean(rel_L1))
            Train_rel_L2_arr.append(np.mean(rel_L2))

            _, _, rel_L1, rel_L2, _ = rel_err(model, test_loader)
            Test_rel_L1_arr.append(np.mean(rel_L1))
            Test_rel_L2_arr.append(np.mean(rel_L2))
        
    t2 = default_timer()
    total_time += t2 - t1

    print(f"Epoch {ep+1}, Time: {t2-t1:.2f}s, Train RMSE: {Train_rmse:.4f}, Test RMSE: {Test_rmse:.4f}")

print(f"total time: {total_time:.2f}")

if SAVE_model:
    torch.save(model.state_dict(), SAVE_model)

if SAVE_lc:
    np.savez(SAVE_lc,
        Train_rel_L1=Train_rel_L1_arr,
        Train_rel_L2=Train_rel_L2_arr,
        Test_rel_L1=Test_rel_L1_arr,
        Test_rel_L2=Test_rel_L2_arr)

Epoch 1, Time: 76.50s, Train RMSE: 0.2517, Test RMSE: 0.2054
Epoch 2, Time: 74.05s, Train RMSE: 0.2289, Test RMSE: 0.1872
Epoch 3, Time: 73.75s, Train RMSE: 0.2200, Test RMSE: 0.1990
Epoch 4, Time: 74.19s, Train RMSE: 0.2129, Test RMSE: 0.1820
Epoch 5, Time: 73.75s, Train RMSE: 0.2062, Test RMSE: 0.1680
Epoch 6, Time: 74.02s, Train RMSE: 0.2013, Test RMSE: 0.1895
Epoch 7, Time: 73.59s, Train RMSE: 0.1983, Test RMSE: 0.2777
Epoch 8, Time: 73.67s, Train RMSE: 0.2062, Test RMSE: 0.1992
Epoch 9, Time: 73.68s, Train RMSE: 0.2040, Test RMSE: 0.1865
Epoch 10, Time: 73.70s, Train RMSE: 0.1915, Test RMSE: 0.1525
Epoch 11, Time: 73.87s, Train RMSE: 0.1872, Test RMSE: 0.1619
Epoch 12, Time: 73.98s, Train RMSE: 0.2008, Test RMSE: 0.1633
Epoch 13, Time: 74.59s, Train RMSE: 0.2004, Test RMSE: 0.1839
Epoch 14, Time: 74.01s, Train RMSE: 0.1825, Test RMSE: 0.1627
Epoch 15, Time: 74.57s, Train RMSE: 0.1854, Test RMSE: 0.1500
Epoch 16, Time: 76.49s, Train RMSE: 0.1862, Test RMSE: 0.1578
Epoch 17, Time: 7